In [26]:
import torch
import pandas as pd
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix, 
    classification_report
)

data = pd.read_csv("../data/synthetic_data.csv")

print(data.head())
print(data.info())
print(data.describe())
print(data.shape)

class IrisNeuralNetwork:
    def __init__(self, input_size, hidden1_size, hidden2_size):
        self.w12 = torch.empty(input_size, hidden1_size)
        self.b1 = torch.empty(hidden1_size)
        self.w23 = torch.empty(hidden1_size, hidden2_size)
        self.b2 = torch.empty(hidden2_size)
        self.w34 = torch.empty(hidden2_size, 1)
        self.b3 = torch.empty(1)

    def sigmoid(self, x):
        return 1 / (1 + torch.exp(-x))

    def forward(self, x):
        z1 = x @ self.w12 + self.b1
        a1 = self.sigmoid(z1)
        z2 = a1 @ self.w23 + self.b2
        a2 = self.sigmoid(z2)
        z3 = a2 @ self.w34 + self.b3
        return self.sigmoid(z3)


history = {
    'train_loss': [],
    'test_loss': [],
    'test_accuracy': []
}

         X1        X2  output
0  0.994416  0.533942       1
1  1.177820 -1.126290       0
2 -0.945938 -0.848435       0
3  0.936008 -0.305729       1
4 -0.694159  0.698876       1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1000 non-null   float64
 1   X2      1000 non-null   float64
 2   output  1000 non-null   int64  
dtypes: float64(2), int64(1)
memory usage: 23.6 KB
None
                X1           X2       output
count  1000.000000  1000.000000  1000.000000
mean     -0.027157    -0.041622     0.489000
std       0.881587     0.867416     0.500129
min      -1.499560    -1.493940     0.000000
25%      -0.776087    -0.756851     0.000000
50%      -0.031119    -0.085191     0.000000
75%       0.767149     0.699188     1.000000
max       1.496600     1.499670     1.000000
(1000, 3)


In [15]:
x = torch.tensor(
    data[["X1", "X2"]].values,
    dtype=torch.float32
)

y = torch.tensor(
    data["output"].values,
    dtype=torch.float32
).reshape(-1, 1)

print(x.shape, y.shape)

N = int(0.8 * len(x))

x_train, x_test = x[:N], x[N:]
y_train, y_test = y[:N], y[N:]

print(f"Training set: {x_train.shape}, {y_train.shape}")
print(f"Test set: {x_test.shape}, {y_test.shape}")

torch.Size([1000, 2]) torch.Size([1000, 1])
Training set: torch.Size([800, 2]), torch.Size([800, 1])
Test set: torch.Size([200, 2]), torch.Size([200, 1])


In [16]:
def binary_cross_entropy(y, prediction):

    # Prevent log(0)
    epsilon = 1e-7

    prediction = torch.clamp(
        prediction,
        epsilon,
        1 - epsilon
    )

    loss = -(
        y * torch.log(prediction)
        +
        (1 - y) * torch.log(1 - prediction)
    )

    return loss.mean()

In [31]:
model = torch.load("../../Iris_neural_net/models/iris_nn_model.pt",
        weights_only=False)

# -----------------------------------------
# Test
# -----------------------------------------

def evaluate(model, x_test, y_test):
    with torch.no_grad():

        probabilities = model.forward(x_test)

        loss = binary_cross_entropy(
            y_test,
            probabilities)

        predictions = (probabilities >= 0.5).float()

        accuracy = (
            predictions == y_test
        ).float().mean()

        print()
        print("Test results:")
        print(
            f"Overall score: {accuracy.item():.7f} | ",
            f"Test Loss: {loss:.6f}"
            )

        print()

        for i in range(len(y_test)):

            actual = int(y_test[i].item())
            predicted = int(predictions[i].item())
            probability = probabilities[i].item()

            print(
                f"{i:03d}: "
                f"actual: {actual} "
                f"predict: {predicted} "
                f"prob: {probability:.7f}"
            )

        return loss.item(), accuracy.item(), predictions, probabilities


test_loss, accuracy, predictions, probabilities = evaluate(
    model, 
    x_test,
    y_test
)

history['train_loss'].append(0.0)  # Placeholder for training loss
history['test_loss'].append(test_loss)
history['test_accuracy'].append(accuracy)



Test results:
Overall score: 0.4800000 |  Test Loss: 2.475302

000: actual: 0 predict: 0 prob: 0.0091868
001: actual: 1 predict: 0 prob: 0.0085808
002: actual: 0 predict: 0 prob: 0.0095423
003: actual: 1 predict: 0 prob: 0.0091715
004: actual: 1 predict: 0 prob: 0.0087130
005: actual: 0 predict: 0 prob: 0.0081067
006: actual: 0 predict: 0 prob: 0.0082086
007: actual: 0 predict: 0 prob: 0.0085691
008: actual: 1 predict: 0 prob: 0.0092574
009: actual: 0 predict: 0 prob: 0.0085151
010: actual: 0 predict: 0 prob: 0.0091440
011: actual: 0 predict: 0 prob: 0.0092192
012: actual: 1 predict: 0 prob: 0.0084507
013: actual: 0 predict: 0 prob: 0.0090616
014: actual: 1 predict: 0 prob: 0.0088743
015: actual: 1 predict: 0 prob: 0.0092169
016: actual: 1 predict: 0 prob: 0.0086807
017: actual: 1 predict: 0 prob: 0.0086108
018: actual: 0 predict: 0 prob: 0.0084675
019: actual: 1 predict: 0 prob: 0.0091629
020: actual: 1 predict: 0 prob: 0.0083850
021: actual: 1 predict: 0 prob: 0.0085660
022: actual:

In [30]:
accuracy = accuracy_score(
    y_test.numpy(),
    predictions.numpy()
)

precision = precision_score(
    y_test.numpy(),
    predictions.numpy()
)

recall = recall_score(
    y_test.numpy(),
    predictions.numpy()
)

f1 = f1_score(
    y_test.numpy(),
    predictions.numpy()
)

cm = confusion_matrix(
    y_test.numpy(),
    predictions.numpy()
)

report = classification_report(
    y_test.numpy(),
    predictions.numpy(),
    output_dict=True
)

print("Confusion Matrix:")
print(cm)
print("Classification Report:")
print(report)

Confusion Matrix:
[[ 96   0]
 [104   0]]
Classification Report:
{'0.0': {'precision': 0.48, 'recall': 1.0, 'f1-score': 0.6486486486486487, 'support': 96.0}, '1.0': {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 104.0}, 'accuracy': 0.48, 'macro avg': {'precision': 0.24, 'recall': 0.5, 'f1-score': 0.32432432432432434, 'support': 200.0}, 'weighted avg': {'precision': 0.2304, 'recall': 0.48, 'f1-score': 0.3113513513513514, 'support': 200.0}}


C:\Users\DanieHKombe\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\DanieHKombe\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\DanieHKombe\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr

In [33]:
experiment = {
    'name': 'INN with Trained Weights',
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1_score': f1,                 
    'history': history,
    'classification_report': report,
    'confusion_matrix': cm
}

In [34]:
import joblib
joblib.dump(
    experiment,
    "../dictionary/trained_weights_experiment.pkl"
)

torch.save(model, "../models/trained_weights.pt")

In [18]:
print("Class 0:", (y_test == 0).sum().item())
print("Class 1:", (y_test == 1).sum().item())

print("\n\n")

print("Predicted 0:", (predictions == 0).sum().item())
print("Predicted 1:", (predictions == 1).sum().item())

Class 0: 96
Class 1: 104



Predicted 0: 200
Predicted 1: 0
